# Acurácia na validação por similaridade

Este notebook cria uma figura para cada modelo de linguagem. Cada figura contém um subplot por dataset e uma curva para cada condição experimental disponível.

A similaridade é dividida em quantis para que cada ponto represente várias questões. O eixo x mostra a similaridade média da faixa e o eixo y mostra a acurácia somente entre respostas resolvidas. A tabela de cobertura deve ser consultada junto com os gráficos.

Condições dos estudantes: baseline, autorreflexão simples/complexa e reflexão externa simples/complexa. Para o GPT-5.4 Petrobras aparecem apenas baseline e as duas condições de autorreflexão, pois ele não possui condições de reflexão externa.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

plt.style.use("seaborn-v0_8-whitegrid")

EXPERIMENT_ID = "91ccab5e5028"
N_SIMILARITY_BINS = 10
SAVE_PDF = True

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "run_experiment.py").exists() and (candidate / "rmcq").is_dir():
            return candidate
    raise FileNotFoundError("Não encontrei a raiz do repositório Reflection-MCQ.")

ROOT = find_repo_root()
RESULT_DIR = ROOT / "data" / "results" / "reflection_top1" / EXPERIMENT_ID
OUTCOMES_PATH = RESULT_DIR / "analysis" / "all_outcomes.jsonl"
PLOTS_DIR = RESULT_DIR / "analysis" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repositório: {ROOT}")
print(f"Resultados:  {OUTCOMES_PATH}")
print(f"Figuras:     {PLOTS_DIR}")

In [ ]:
if not OUTCOMES_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {OUTCOMES_PATH}\n"
        "Execute o estágio finish no servidor GPU ou copie a pasta final de resultados."
    )

results = pd.read_json(OUTCOMES_PATH, lines=True)
required_columns = {"model", "dataset", "condition", "val_uid", "similarity", "correct"}
missing_columns = sorted(required_columns - set(results.columns))
if missing_columns:
    raise ValueError(f"Colunas ausentes em all_outcomes.jsonl: {missing_columns}")

results["similarity"] = pd.to_numeric(results["similarity"], errors="coerce")
results["correct_value"] = results["correct"].map({True: 1.0, False: 0.0})

print(f"Linhas: {len(results):,}")
print(f"Modelos: {', '.join(map(str, results['model'].dropna().unique()))}")
print(f"Datasets: {', '.join(map(str, results['dataset'].dropna().unique()))}")
results.head(3)

In [ ]:
CONDITION_ORDER = [
    "baseline",
    "self_simple",
    "self_complex",
    "teacher_simple",
    "teacher_complex",
]

CONDITION_LABELS = {
    "baseline": "Baseline",
    "self_simple": "Self-ref. simples",
    "self_complex": "Self-ref. complexa",
    "teacher_simple": "Ref. externa simples",
    "teacher_complex": "Ref. externa complexa",
}

CONDITION_COLORS = {
    "baseline": "#222222",
    "self_simple": "#1f77b4",
    "self_complex": "#17becf",
    "teacher_simple": "#d62728",
    "teacher_complex": "#ff7f0e",
}

DATASET_ORDER = ["aqua", "arc", "logiqa2", "openbookqa"]
DATASET_LABELS = {
    "aqua": "AQuA",
    "arc": "ARC",
    "logiqa2": "LogiQA 2.0",
    "openbookqa": "OpenBookQA",
}

MODEL_LABELS = {
    "phi2": "Phi-2",
    "deepseek-r1-distill-llama-8b": "DeepSeek-R1-Distill-Llama-8B",
    "llama3.1-8b": "Llama 3.1 8B",
    "gpt-5-4-petrobras": "GPT-5.4 Petrobras",
}

# Uma única atribuição de faixas por dataset mantém condições e modelos alinhados.
pair_similarity = (
    results.loc[results["similarity"].notna(), ["dataset", "val_uid", "similarity"]]
    .drop_duplicates(["dataset", "val_uid"])
    .copy()
)

def assign_quantile_bins(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    n_bins = min(N_SIMILARITY_BINS, group["similarity"].nunique(), len(group))
    if n_bins < 2:
        group["similarity_bin"] = 0
    else:
        group["similarity_bin"] = pd.qcut(
            group["similarity"], q=n_bins, labels=False, duplicates="drop"
        ).astype(int)
    return group

pair_similarity = pd.concat(
    [
        assign_quantile_bins(group.drop(columns="dataset")).assign(dataset=dataset)
        for dataset, group in pair_similarity.groupby("dataset", sort=False)
    ],
    ignore_index=True,
)

plot_rows = results.merge(
    pair_similarity[["dataset", "val_uid", "similarity_bin"]],
    on=["dataset", "val_uid"],
    how="inner",
    validate="many_to_one",
)

bin_centers = (
    pair_similarity.groupby(["dataset", "similarity_bin"], as_index=False)
    .agg(similarity=("similarity", "mean"), n_questions=("val_uid", "nunique"))
)

binned_accuracy = (
    plot_rows.groupby(["model", "dataset", "condition", "similarity_bin"], as_index=False)
    .agg(accuracy=("correct_value", "mean"), n_resolved=("correct_value", "count"))
    .merge(bin_centers, on=["dataset", "similarity_bin"], how="left")
)

binned_accuracy.head()

In [ ]:
coverage = (
    plot_rows.groupby(["model", "dataset", "condition"], as_index=False)
    .agg(
        n=("val_uid", "size"),
        resolved=("correct_value", "count"),
        accuracy=("correct_value", "mean"),
    )
)
coverage["coverage"] = coverage["resolved"] / coverage["n"]
coverage["condition"] = pd.Categorical(
    coverage["condition"], categories=CONDITION_ORDER, ordered=True
)
coverage = coverage.sort_values(["model", "dataset", "condition"])
coverage.style.format({"accuracy": "{:.1%}", "coverage": "{:.1%}"})

In [ ]:
def conditions_for_model(model: str) -> list[str]:
    available = set(plot_rows.loc[plot_rows["model"] == model, "condition"])
    if model == "gpt-5-4-petrobras" or model.casefold().startswith("gpt-5"):
        desired = ["baseline", "self_simple", "self_complex"]
    else:
        desired = CONDITION_ORDER
    return [condition for condition in desired if condition in available]

def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")

def plot_model(model: str, save: bool = True):
    model_data = binned_accuracy[binned_accuracy["model"] == model]
    available_datasets = set(model_data["dataset"])
    datasets = [d for d in DATASET_ORDER if d in available_datasets]
    datasets += sorted(available_datasets - set(datasets))
    conditions = conditions_for_model(model)

    if not datasets:
        raise ValueError(f"Não há resultados com similaridade para o modelo {model!r}.")

    ncols = 2
    nrows = math.ceil(len(datasets) / ncols)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(13, 4.6 * nrows), sharey=True, squeeze=False
    )
    legend_handles = {}

    for ax, dataset in zip(axes.flat, datasets):
        subset = model_data[model_data["dataset"] == dataset]
        for condition in conditions:
            line_data = subset[subset["condition"] == condition].sort_values("similarity")
            line_data = line_data[line_data["accuracy"].notna()]
            if line_data.empty:
                continue
            line, = ax.plot(
                line_data["similarity"],
                line_data["accuracy"],
                color=CONDITION_COLORS[condition],
                marker="o",
                markersize=4.5,
                linewidth=2,
                label=CONDITION_LABELS[condition],
            )
            legend_handles[condition] = line

        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xlabel("Similaridade média no quantil")
        ax.set_ylabel("Acurácia")
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(alpha=0.25)

    for ax in axes.flat[len(datasets):]:
        ax.set_visible(False)

    ordered_handles = [legend_handles[c] for c in conditions if c in legend_handles]
    ordered_labels = [CONDITION_LABELS[c] for c in conditions if c in legend_handles]
    if ordered_handles:
        fig.legend(
            ordered_handles, ordered_labels, loc="lower center", ncol=len(ordered_handles),
            bbox_to_anchor=(0.5, 0.01), frameon=False,
        )

    title = MODEL_LABELS.get(model, model)
    fig.suptitle(f"{title}: acurácia na validação por similaridade", fontsize=15, y=0.995)
    fig.tight_layout(rect=(0, 0.075, 1, 0.97))

    if save:
        png_path = PLOTS_DIR / f"accuracy_by_similarity_{slug(model)}.png"
        fig.savefig(png_path, dpi=180, bbox_inches="tight")
        if SAVE_PDF:
            fig.savefig(png_path.with_suffix(".pdf"), bbox_inches="tight")
        print(f"Salvo: {png_path}")

    plt.show()
    return fig


In [ ]:
preferred_models = [
    "phi2",
    "deepseek-r1-distill-llama-8b",
    "llama3.1-8b",
    "gpt-5-4-petrobras",
]
available_models = list(map(str, results["model"].dropna().unique()))
model_order = [model for model in preferred_models if model in available_models]
model_order += sorted(set(available_models) - set(model_order))

figures = {}
for model in model_order:
    figures[model] = plot_model(model)

## Leitura dos gráficos

- Uma curva mais alta indica maior acurácia naquela faixa de similaridade.
- Compare curvas dentro do mesmo subplot; os quantis são compartilhados entre modelos e condições de um dataset.
- Pontos são calculados apenas com respostas resolvidas. Verifique a tabela de cobertura antes de interpretar diferenças de acurácia.
- Oscilações em AQuA e ARC podem ser maiores porque esses conjuntos têm menos questões de validação.